# Portfolio: Building a Runner Script for My EEG Connectivity Pipeline
**Author:** Cynthia Deng  
**Date** 03-10-2025

---

## Goal

This notebook documents my learning process as I figured out how to write a **runner script** for my EEG connectivity pipeline. I came into this knowing basic Python: variables, loops, functions, that kind of thing. But I had never written a script that was specifically designed to launch another script or pipeline.

Throughout this portfolio, I used **Claude** as a learning tool. I would paste code I did not understand, ask what it does, and then try to restate the explanation in my own words. I have noted throughout where Claude helped me. The goal was not to have Claude write the portfolio for me. I used it more like a tutor: someone I could ask questions to and then figure out the answer myself.

The final script I am documenting is called `aging_stim_run_.py`. It runs an EEG connectivity analysis across multiple subjects and two task conditions.

## Step 1: Why Do We Even Need a Separate Runner File?

The first thing I was confused about was: *why does this script exist at all?* I already had a pipeline file called `aging_pipeline.py` that had all the actual analysis code in it. Why could I not just run that directly?

I asked Claude this question and the answer made a lot of sense once I heard it. Claude explained that the pipeline file is like a toolbox. It defines what can be done: the functions, the methods, the logic. The runner file is the instruction sheet. It says what to actually do, with which data, and in what order.

Here is a concrete way I thought about it. If I have 20+ subjects and 2 conditions, I could technically open the pipeline file and manually call every function one by one. But that would be exhausting and easy to mess up. The runner file automates all of that. It is the script that says: go through every condition, call the pipeline on every subject, and save the output.

Claude also mentioned keeping code modular as another reason. If I want to change a parameter, like the time window I am analyzing, I change it in one place in the runner file rather than trying to find where it lives inside the pipeline. That kind of separation makes the whole project easier to manage over time.

This was my first real "oh, that makes sense" moment.

## Step 2: Understanding the Imports

The first actual lines of code in the runner file are the imports. I knew what `import` meant at a basic level, but I did not always know *why* each specific import was there. Let me go through them.

In [ ]:
import pandas as pd
from pathlib import Path
from aging_pipeline import compute_fp_ciPLV_group, SUBJECT_IDS

**`import pandas as pd`** and **`from pathlib import Path`**  
These are utility libraries. `pandas` is for organizing data into tables and saving them as CSV files. `Path` is for handling file paths in a clean and reliable way. I was already somewhat familiar with both and did not need to dig too deep here.

**`from aging_pipeline import compute_fp_ciPLV_group, SUBJECT_IDS`**  
This was the most important import to understand. I am pulling two specific things from my pipeline file:

- `compute_fp_ciPLV_group` is the main analysis function that processes EEG data for a group of subjects
- `SUBJECT_IDS` is the list of subject ID numbers defined in the pipeline file

Claude helped me understand that using `from ... import ...` is a way of being explicit about what the runner file depends on. Instead of importing the entire pipeline module, I am only bringing in what I actually need. It also makes the code easier to read because anyone looking at the runner file immediately knows which pieces of the pipeline are being used.

## Step 3: Setting Up Conditions and Directories

The next section defines the conditions to process and the folder paths for where the data lives and where the results should go.

In [ ]:
CONDITIONS = ["ontask", "offtask"]

BASE_DIR = Path("/Users/CynthiaDeng/Downloads/aging_data")
OUT_DIR  = Path("/Users/CynthiaDeng/Desktop/source_output_aging_stimlocked")

**`CONDITIONS = ["ontask", "offtask"]`**  
This is a list with two strings representing the two experimental conditions in my study. "ontask" means the participant was actively doing a cognitive task when the EEG was recorded. "offtask" means they were in a more passive or resting state. By storing them in a list, the script can loop through both conditions automatically rather than me having to copy and paste the analysis code twice.

I asked Claude why this variable is written in ALL CAPS. It told me this is a Python convention for **constants**, meaning variables whose values are not supposed to change while the script runs. Writing them in ALL_CAPS is a signal to anyone reading the code: this is a fixed setting, do not change it inside a loop or function. Python itself does not enforce this, it is just a widely used style convention.

**`BASE_DIR` and `OUT_DIR`**  
These point to the two key folders the script cares about. `BASE_DIR` is where my raw EEG data files are stored. `OUT_DIR` is where the processed results will be saved.

Putting these at the very top of the script is intentional. If I ever move my data to a different folder, I only have to update one line. Claude called this "centralizing your configuration" and said it is a good habit for any script that reads or writes files.

## Step 4: Pipeline Parameters and Why We Re-declare Them Here

I knew that my pipeline file (`aging_pipeline.py`) already has these parameters defined inside the function signature. So why does the runner file need to list them again?

In [ ]:
n_surrogates  = 500      # surrogate draws per subject
subsample_n   = 94       # trials per draw (min across all subjects)
random_state  = 0
tmin = -0.2              # epoch window start (seconds)
tmax = 0.8               # epoch window end (seconds)
conn_tmin = 0.0          # connectivity computed from 0 ms
conn_tmax = 0.6          # connectivity computed to 600 ms

I asked Claude about this directly and got a clear answer. The values inside the pipeline function are **default values**, kind of like factory settings. They are what the function will use if no one tells it otherwise. The runner file is where I say: no, for this specific study, I want to use *these* values. By re-declaring them here, the runner is explicitly overriding the defaults.

There is also a practical reason. If I want to change a parameter, like making the connectivity window longer, I change it here in the runner file. I do not have to go digging through the pipeline code. Everything that can be tuned for a given analysis run lives in one visible place at the top of this script. That makes the whole thing much easier to manage.

Now here is what each parameter actually means:

**`n_surrogates = 500`**  
Surrogate analysis is a statistical method used in EEG connectivity research to test whether the connectivity we measure is real or just random noise. We generate 500 shuffled versions of the data and compare our actual result against that distribution. 500 is enough draws to get a stable statistical estimate without the script taking too long to run.

**`subsample_n = 94`**  
Not every subject has the same number of clean trials after artifact rejection. To make a fair comparison, the script randomly draws the same number of trials from each subject on each surrogate run. 94 comes from the subject with the fewest usable trials, so nobody gets excluded from the analysis.

**`random_state = 0`**  
I asked Claude why this is set to 0 specifically. It explained that this seeds the random number generator. By fixing the seed, the random subsampling happens in the same order every single time you run the script. Without this, results would vary slightly each run and the analysis would not be reproducible. The value 0 is just conventional; any fixed number would work.

**`tmin = -0.2` and `tmax = 0.8`**  
These define the time window for each EEG epoch relative to when the stimulus appeared. We start 200 ms before the stimulus (negative = before) and end 800 ms after. The pre-stimulus window is important because it is used for **baseline correction**, which means subtracting out whatever the brain was doing before the stimulus so that we can isolate the actual response to it.

**`conn_tmin = 0.0` and `conn_tmax = 0.6`**  
Even though we record the full epoch from -200 ms to 800 ms, the connectivity analysis only runs from 0 to 600 ms. The comment in the code mentions this is to "avoid motor" activity. If participants pressed a button in response to the stimulus, that motor signal would show up in the EEG and contaminate the connectivity measure. Stopping at 600 ms keeps the analysis focused on the cognitive response to the stimulus rather than the physical button press.

## Step 5: The `if __name__ == "__main__":` Block

Honestly, this was the line that confused me the most when I first saw it. *if the name is main?* What does that even mean?

I asked Claude to explain this multiple times in different ways before it finally clicked.

In [ ]:
if __name__ == "__main__":

    import aging_pipeline
    print(">>> USING aging_pipeline:", aging_pipeline.__file__)

Here is what I now understand. In Python, every script has a built-in variable called `__name__`. When you run a script directly from the terminal (for example, `python aging_stim_run_.py`), Python automatically sets `__name__` to the string `"__main__"`. But if this same file were imported by some other script, `__name__` would be set to the file's own name instead.

So the line `if __name__ == "__main__":` is basically saying: only run everything below this point if this script is being executed directly, not if it is being imported somewhere else.

Why does that matter? Because the runner script is meant to be *run*, not imported. This guard makes sure the full analysis does not accidentally trigger if someone loads this file as a module in another project.

The two lines inside this block import the pipeline module and then immediately print the file path of the version being used. Claude pointed out that this is a **sanity check**. If I had multiple versions of `aging_pipeline.py` sitting on my computer, this print statement would tell me right away which one actually got loaded. That kind of verification step is a really practical habit in research code where file versions can easily get mixed up.

This was a big conceptual jump for me and I had to sit with it for a while. But once it clicked, I started seeing this pattern in almost every serious Python project I looked at.

## Step 6: The Main Loop

This is the core of the runner script. It is where the actual work gets done.

In [ ]:
    for condition in CONDITIONS:

        print(f"\n==========================================")
        print(f">>> Processing condition: {condition}")

        try:
            results = compute_fp_ciPLV_group(
                subject_ids   = SUBJECT_IDS,
                eeglab_dir    = str(BASE_DIR),
                condition     = condition,
                n_surrogates  = n_surrogates,
                subsample_n   = subsample_n,
                random_state  = random_state,
                tmin          = tmin,
                tmax          = tmax,
                conn_tmin     = conn_tmin,
                conn_tmax     = conn_tmax,
            )

        except Exception as e:
            print(f"  ERROR for condition={condition}: {e}")
            continue

**The `for` loop**  
This loops through `["ontask", "offtask"]`. On the first pass, `condition` equals `"ontask"`. On the second pass, it equals `"offtask"`. The power here is that all the analysis code is written only once and the loop handles running it for both conditions.

**The `print` statements**  
These are progress markers. EEG analysis can take a long time to run, so having the terminal print out where the script currently is helps me know it is still working and has not frozen. The `\n` just adds a blank line before the separator to make the output easier to read. The `f"...{condition}..."` syntax is an f-string, which inserts the current value of `condition` directly into the printed text.

**`try` and `except Exception as e`**  
I had seen this pattern before but never really understood why it was needed in this context. I asked Claude: why not just call the function directly?

The reason is robustness. If `compute_fp_ciPLV_group` crashes on the "ontask" condition, maybe because a file is missing or the data is corrupted, the script would normally stop completely. With `try/except`, the error message gets printed to the terminal and then `continue` tells the loop to move on to the next condition instead of halting everything. For a long analysis run with multiple subjects and conditions, this protection is really important. I do not want to lose all the "offtask" results just because "ontask" had a problem.

`as e` stores the actual error message so it gets printed. That way I can see *what* went wrong, not just that something failed.

## Step 7: Why We Save to a CSV File

Before walking through the saving code, I want to explain something I actually had to think about. Stefon told me to save the output to a csv, but Why do we need to save to a CSV at all? What happens if we just let the script finish without saving anything?

In [ ]:
        if not results["fp_summary"]:
            print(f"  No results for {condition}, skipping CSV save.")
            continue

        # Save one CSV per condition
        out_csv = OUT_DIR / f"fp_summary_{condition}.csv"
        out_csv.parent.mkdir(parents=True, exist_ok=True)

        df = pd.DataFrame(results["fp_summary"])
        preferred_cols = ["sub", "condition", "n_epochs", "lh_fp", "rh_fp", "fp_mean"]
        existing  = [col for col in preferred_cols if col in df.columns]
        remaining = [col for col in df.columns if col not in existing]
        df = df[existing + remaining]
        df.to_csv(out_csv, index=False)

        print(f">>> Saved: {out_csv} ({len(df)} rows)")

**What happens if we do not save**  
The `results` variable exists only in the computer's memory while the script is running. The moment the script finishes, everything stored in that variable disappears. To look at any number again, I would have to re-run the entire pipeline from scratch. For EEG source analysis with surrogate resampling across 40+ subjects, that could take many hours. Without saving, all of that computation is just gone.

**What we get by saving to a CSV**  
Once saved, the results are on disk permanently. I can open the file in Excel to take a quick look. I can load it into R or Python for statistical analysis. I can send it to my advisor. I never have to re-run the heavy computation just to see a number. The CSV is also a clean, readable format that works with almost every analysis tool, which makes it a solid default choice for research outputs.

**`if not results["fp_summary"]:`**  
Before trying to save anything, we check whether the results dictionary actually contains data. If the pipeline ran but came back empty (for example, no subjects were processed successfully), there is nothing to save. This prevents the script from writing a broken or empty CSV file.

**`out_csv = OUT_DIR / f"fp_summary_{condition}.csv"`**  
This builds the full file path for the output. For the "ontask" condition it would produce something like `/Users/CynthiaDeng/Desktop/source_output_aging_stimlocked/fp_summary_ontask.csv`.

**`out_csv.parent.mkdir(parents=True, exist_ok=True)`**  
This was a line I specifically asked Claude about. `out_csv.parent` refers to the folder the file will be saved in. `.mkdir(parents=True, exist_ok=True)` creates that folder if it does not exist yet. The `exist_ok=True` part means it will not crash if the folder already exists. Without this line, the script would throw an error any time the output folder had not been created manually beforehand.

**Building the DataFrame and ordering columns**  
```python
df = pd.DataFrame(results["fp_summary"])
preferred_cols = ["sub", "condition", "n_epochs", "lh_fp", "rh_fp", "fp_mean"]
existing  = [col for col in preferred_cols if col in df.columns]
remaining = [col for col in df.columns if col not in existing]
df = df[existing + remaining]
```
This converts the results into a pandas DataFrame and then reorders the columns so the most important ones appear first. `preferred_cols` is the desired column order. `existing` filters that list down to only the columns that actually exist in the data, in case some are missing. `remaining` collects any extra columns not in the preferred list. The final line combines them: preferred columns first, then anything extra.

I asked Claude why we bother reordering instead of just writing the CSV directly. It said this is a practical choice: when I open the file in Excel later, I want subject ID, condition, and connectivity values to show up immediately without having to scroll or rearrange anything.

**`df.to_csv(out_csv, index=False)`**  
The `index=False` argument stops pandas from adding a numbered row index (0, 1, 2...) as an extra column in the CSV. By default pandas includes that index, but it would just be noise in the final file since the subject ID column already uniquely identifies each row.

## Step 8: The Complete Script

Now that I have walked through every section, here is the full runner script in one place. Reading it now feels very different from the first time I saw it.

In [ ]:
# aging_stim_run_.py
# Runs ALL subjects across BOTH conditions (ontask, offtask)
# Saves ONE combined CSV per condition

import pandas as pd
from pathlib import Path
from aging_pipeline import compute_fp_ciPLV_group, SUBJECT_IDS

# --------------------------------------------------
# CONDITIONS TO PROCESS
# --------------------------------------------------
CONDITIONS = ["ontask", "offtask"]

BASE_DIR = Path("/Users/CynthiaDeng/Downloads/aging_data")
OUT_DIR  = Path("/Users/CynthiaDeng/Desktop/source_output_aging_stimlocked")

# --------------------------------------------------
# Pipeline parameters
# --------------------------------------------------
n_surrogates  = 500
subsample_n   = 94
random_state  = 0
tmin = -0.2
tmax = 0.8
conn_tmin = 0.0
conn_tmax = 0.6

if __name__ == "__main__":

    import aging_pipeline
    print(">>> USING aging_pipeline:", aging_pipeline.__file__)

    for condition in CONDITIONS:

        print(f"\n==========================================")
        print(f">>> Processing condition: {condition}")

        try:
            results = compute_fp_ciPLV_group(
                subject_ids   = SUBJECT_IDS,
                eeglab_dir    = str(BASE_DIR),
                condition     = condition,
                n_surrogates  = n_surrogates,
                subsample_n   = subsample_n,
                random_state  = random_state,
                tmin          = tmin,
                tmax          = tmax,
                conn_tmin     = conn_tmin,
                conn_tmax     = conn_tmax,
            )

        except Exception as e:
            print(f"  ERROR for condition={condition}: {e}")
            continue

        if not results["fp_summary"]:
            print(f"  No results for {condition}, skipping CSV save.")
            continue

        out_csv = OUT_DIR / f"fp_summary_{condition}.csv"
        out_csv.parent.mkdir(parents=True, exist_ok=True)

        df = pd.DataFrame(results["fp_summary"])
        preferred_cols = ["sub", "condition", "n_epochs", "lh_fp", "rh_fp", "fp_mean"]
        existing  = [col for col in preferred_cols if col in df.columns]
        remaining = [col for col in df.columns if col not in existing]
        df = df[existing + remaining]
        df.to_csv(out_csv, index=False)

        print(f">>> Saved: {out_csv} ({len(df)} rows)")

    print("\n>>> ALL DONE")

## Step 9: Reflection

When I first looked at this script, it felt overwhelming. Not because any single line was impossible to understand, but because I could not see the *purpose* behind the structure. Why a separate file? Why all caps? Why `try/except`? Why does `if __name__ == "__main__":` exist?

Going through it section by section, mostly by asking Claude questions and then restating the answers in my own words, changed how I read Python code. I started to see that most of the lines I originally thought were just extra or boilerplate are actually deliberate decisions. They make the script more robust, more readable, and more reproducible.

The biggest things I took away:

1. **The pipeline and the runner serve different purposes.** The pipeline does the science. The runner does the logistics. Keeping them separate makes both easier to work with.

2. **Re-declaring parameters in the runner is not redundant.** The pipeline has default values, but the runner is where I explicitly set what I actually want for this specific study. It also means everything tunable lives in one place.

3. **Saving to CSV is not optional for real research.** If I do not save, the results exist only in memory and disappear the moment the script ends. Hours of computation would have to be redone just to see a single number.

4. **Error handling protects long runs.** With `try/except/continue`, one bad condition does not take down the entire analysis.

5. **`if __name__ == "__main__":` is a guard, not just a habit.** It protects the script from running accidentally if someone imports it, which is more common than I expected in research projects.

I used Claude throughout this process the way I would use a knowledgeable tutor. I would paste code, ask what it does, and then write the explanation in my own words until I actually understood it. The explanations in this notebook are mine. Claude helped me get there, but the understanding is something I now genuinely have.

---
*Tools used: Python 3, Jupyter Notebook, Claude (Anthropic) as a learning assistant*